# OpenWebUI Trace PromptFoo Optimization Experiments

This notebook is a guided how-to for `trace_openwebui_dea_skeleton.py`. It helps non-expert users choose what to optimize, build bounded experiment commands, run them when explicitly enabled, and compare scores, parameters, and timings.

By default it is safe: it only builds and displays a plan. Set `RUN_OPENWEBUI_TRACE_EXPERIMENTS=1` to execute experiments.

## How To Choose What To Optimize

| Goal | Use | Configure |
|---|---|---|
| Model only, no tools | `optimize_target="model"` | `TARGET_MODEL_ID`, `SYSTEM_PROMPT`, `MODEL_PARAMS` |
| System prompt focused run | `PROFILE="model_system_prompt_only"` | `SYSTEM_PROMPT`; keep `MODEL_PARAMS` minimal |
| Sampling/length params | `PROFILE="model_sampling_params"` | `generation_temperature`, `generation_top_p`, `generation_max_tokens` |
| Provider/model-specific params | `PROFILE="model_extra_params"` | `model_extra_payload_json`, for example `frequency_penalty`, `think`, or provider-specific `reasoning` |
| Tool/pipe params | `optimize_target="tool"` | `TOOL_TARGET_MODEL_ID`, `TOOL_PARAMS` |
| PromptFoo Offline Process | `PROFILE="promptfoo_model_offline"` or `PROFILE="promptfoo_tool_offline"` | OpenWebUI generates first, PromptFoo scores the saved CSV |
| PromptFoo Online Process | `PROFILE="promptfoo_model_online"` or `PROFILE="promptfoo_tool_online"` | PromptFoo calls the OpenWebUI bridge live through its provider config |
| Compare methods | `PROFILE="method_matrix"` | Runs direct DEA, DEA judge, LLM judge, Offline Process, and Online Process examples |

`model_extra_payload_json` is intentionally configurable. It is forwarded to OpenWebUI/OpenAI-compatible payloads, but the target model/provider may ignore keys it does not support.

## Using Any PromptFoo Evaluation

DEA configs are examples, not requirements. The Trace harness can optimize an OpenWebUI model or tool using any PromptFoo config as long as the PromptFoo command writes a JSON result to `{result_json}`. That JSON can be PromptFoo's normal output, a red-team report adapted to a score JSON, or a custom wrapper with a top-level `score` or nested row records containing `score` and optional `namedScores`.

| User-facing process | Internal method | What happens | Typical PromptFoo config |
|---|---|---|---|
| Offline Process | `step2_promptfoo` | Trace calls OpenWebUI first, writes `candidate_answer` into a temporary CSV, then PromptFoo evaluates that CSV | `echo` provider or assertions over `{{candidate_answer}}` |
| Online Process | `step3_promptfoo` | Trace prepares rows, then PromptFoo calls the OpenWebUI bridge live through its provider config and evaluates returned output | HTTP provider calling `/generate` or another live provider |

For a custom CSV, provide at least one prompt column: `request_prompt`, `prompt`, `question`, `input`, `query`, or `instruction`. Extra columns such as `expected`, `reference`, `attack_category`, `policy`, or rubric fields are preserved for PromptFoo assertions. For JSON or JSONL datasets, set `BUILD_INPUT_CSV_CMD` to a command that creates the CSV path passed as `{output_csv}`.

## Objective Evidence Matrix

| Requirement | Current evidence | Runtime caveat |
|---|---|---|
| Optimize model parameters | `OpenWebUIModel`, model trainer tests, model notebook profiles | Live quality depends on target model availability and speed |
| Optimize tool parameters | `OpenWebUISummarizerAgent`, tool forwarding tests, tool notebook profiles | The OpenWebUI pipe/tool must consume the forwarded params |
| Offline Process | Trainer test covers bridge generation followed by PromptFoo on `step2_candidate_rows.csv` | Full live generation depends on OpenWebUI/backend health |
| Online Process | Trainer test covers prepared live rows and PromptFoo scoring artifacts | PromptFoo response transforms can hide bridge internals |
| Generic PromptFoo configs | PromptFoo result parsing accepts top-level scores or nested row scores | Red-team reports may need an adapter command to write `{result_json}` |
| Direct DEA with DEA judge | DEA guide test verifies `use_dea_judge=True` forwarding | Real judge latency/cost depends on configured judge backend |
| Trace and feedback | Step artifacts include summaries, traces, candidates, feedback, scores, params, durations | Online Process records PromptFoo command/runtime unless trace sidecars are added |
| Example tool models | The `summarizer---...` IDs are examples from this OpenWebUI setup | Replace them with your own model/tool IDs as needed |

In [ ]:
# Purpose: imports plus one user-editable configuration block for all later cells.
from __future__ import annotations

import json
import os
import shlex
import subprocess
import time
from datetime import datetime
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except Exception:
    pd = None

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    """Return the repository root containing the Trace optimization script."""
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts/promptfoo_openwebui_eval/trace_openwebui_dea_skeleton.py').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


def env_bool(name: str, default: bool = False) -> bool:
    """Read a boolean environment flag using common truthy strings."""
    raw = os.environ.get(name, '').strip().lower()
    if not raw:
        return default
    return raw in {'1', 'true', 'yes', 'on'}


def env_json(name: str, default: dict[str, Any]) -> dict[str, Any]:
    """Read a JSON object from an environment variable or return defaults."""
    raw = os.environ.get(name, '').strip()
    if not raw:
        return dict(default)
    parsed = json.loads(raw)
    if not isinstance(parsed, dict):
        raise ValueError(f'{name} must contain a JSON object')
    return parsed


def rel(path: Path | str) -> str:
    """Render a path relative to the repository when possible."""
    path = Path(path)
    try:
        return str(path.resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(path)


def show_table(rows: list[dict[str, Any]], title: str = '') -> None:
    """Display rows as a DataFrame, falling back to JSON markdown."""
    if title:
        display(Markdown(f'### {title}'))
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        display(Markdown('```json\n' + json.dumps(rows, indent=2, ensure_ascii=False) + '\n```'))


# REPO_ROOT locates the repository. Set TRACE_REPO_ROOT when running the notebook from outside the repo.
REPO_ROOT = Path(os.environ['TRACE_REPO_ROOT']).resolve() if os.environ.get('TRACE_REPO_ROOT') else find_repo_root(Path.cwd())

# BUNDLE is the PromptFoo/OpenWebUI evaluation bundle directory inside the repo.
BUNDLE = REPO_ROOT / 'scripts/promptfoo_openwebui_eval'

# SCRIPT is the optimization harness this notebook drives.
SCRIPT = BUNDLE / 'trace_openwebui_dea_skeleton.py'

# DATASET_NAME selects the bundled dataset name used for default CSV/config paths.
DATASET_NAME = os.environ.get('TRACE_DATASET_NAME', 'bigsurvey')

# INPUT_CSV is the dataset rows to optimize/evaluate; override with TRACE_INPUT_CSV for custom CSVs.
INPUT_CSV = Path(os.environ.get('TRACE_INPUT_CSV', BUNDLE / 'datasets' / DATASET_NAME / 'step2_input.csv')).resolve()

# RESULTS_DIR stores plans, summaries, logs, and per-step artifacts; override TRACE_EXPERIMENT_RUN_ID for reproducible folders.
RUN_ID = os.environ.get('TRACE_EXPERIMENT_RUN_ID') or datetime.now().strftime('%Y%m%d_%H%M%S')
RESULTS_DIR = BUNDLE / 'results' / 'trace_experiments' / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# RUN_LIVE controls execution. Leave false for planning only; set RUN_OPENWEBUI_TRACE_EXPERIMENTS=1 to run commands.
RUN_LIVE = env_bool('RUN_OPENWEBUI_TRACE_EXPERIMENTS', False)

# PROFILE chooses a recipe group defined below; override with TRACE_EXPERIMENT_PROFILE.
PROFILE = os.environ.get('TRACE_EXPERIMENT_PROFILE', 'model_system_prompt_only')

# BATCH_SIZE is the number of examples per optimization feedback step.
BATCH_SIZE = int(os.environ.get('TRACE_BATCH_SIZE', '3'))

# NUM_EPOCHS is the number of Trace trainer passes over the selected batches.
NUM_EPOCHS = int(os.environ.get('TRACE_NUM_EPOCHS', '1'))

# BRIDGE_URL is the local bridge endpoint used by live bridge generation.
BRIDGE_URL = os.environ.get('TRACE_BRIDGE_URL', 'http://127.0.0.1:8001/generate')

# TARGET_MODEL_ID is the direct OpenWebUI/base/workspace model for model-only optimization.
TARGET_MODEL_ID = os.environ.get('TRACE_TARGET_MODEL_ID', 'qwen-laptop:latest')

# TOOL_TARGET_MODEL_ID is an example OpenWebUI pipe/tool model; replace it with your own setup-specific tool model.
TOOL_TARGET_MODEL_ID = os.environ.get('TRACE_TOOL_TARGET_MODEL_ID', 'summarizer---kohaku')

# SYSTEM_PROMPT is the starting model system prompt for model optimization profiles.
SYSTEM_PROMPT = os.environ.get('TRACE_SYSTEM_PROMPT', 'Answer with concise, evidence-grounded synthesis.')

# MODEL_PARAMS are standard model generation params plus optional provider-specific model_extra_payload_json.
MODEL_PARAMS = env_json('TRACE_MODEL_PARAMS_JSON', {'generation_temperature': 0.2, 'generation_top_p': 0.95, 'generation_max_tokens': 4096})

# MODEL_EXTRA_PAYLOAD is merged into MODEL_PARAMS when using the model_extra_params profile.
MODEL_EXTRA_PAYLOAD = env_json('TRACE_MODEL_EXTRA_PAYLOAD_JSON', {'frequency_penalty': 0.0, 'think': False})

# TOOL_PARAMS are example tool/pipe params; replace keys with what your OpenWebUI tool exposes.
TOOL_PARAMS = env_json('TRACE_TOOL_PARAMS_JSON', {'algorithm': 'kohaku', 'target_length': 'long', 'structure': 'thematic'})

# PROMPTFOO_WORKDIR is where PromptFoo resolves config-relative files.
PROMPTFOO_WORKDIR = Path(os.environ.get('TRACE_PROMPTFOO_WORKDIR', BUNDLE)).resolve()

# PROMPTFOO_OFFLINE_CONFIG is used by the Offline Process, where Trace generates answers before PromptFoo scores them.
PROMPTFOO_OFFLINE_CONFIG = os.environ.get('TRACE_PROMPTFOO_OFFLINE_CONFIG', f'{DATASET_NAME}.step2.dea.yaml')

# PROMPTFOO_ONLINE_CONFIG is used by the Online Process, where PromptFoo calls the bridge live.
PROMPTFOO_ONLINE_CONFIG = os.environ.get('TRACE_PROMPTFOO_ONLINE_CONFIG', f'{DATASET_NAME}.step3.dea.yaml')

# PROMPTFOO_EVAL_CMD must write PromptFoo-compatible JSON to {result_json}; adapter commands are allowed.
PROMPTFOO_EVAL_CMD = os.environ.get('TRACE_PROMPTFOO_EVAL_CMD', 'promptfoo eval -c {config} -t {csv} --output {result_json} --no-progress-bar')

# BUILD_INPUT_CSV_CMD can create INPUT_CSV from JSON/JSONL; use {output_csv} as the destination placeholder.
BUILD_INPUT_CSV_CMD = os.environ.get('TRACE_BUILD_INPUT_CSV_CMD', '').strip()

# FORCE_BUILD_INPUT_CSV rebuilds INPUT_CSV even if it already exists.
FORCE_BUILD_INPUT_CSV = env_bool('TRACE_FORCE_BUILD_INPUT_CSV', False)

config_row = {
    'repo_root': '.',
    'bundle': rel(BUNDLE),
    'dataset_name': DATASET_NAME,
    'input_csv': rel(INPUT_CSV),
    'results_dir': rel(RESULTS_DIR),
    'run_live': RUN_LIVE,
    'profile': PROFILE,
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'target_model_id': TARGET_MODEL_ID,
    'tool_target_model_id': TOOL_TARGET_MODEL_ID,
    'promptfoo_workdir': rel(PROMPTFOO_WORKDIR),
    'promptfoo_offline_config': PROMPTFOO_OFFLINE_CONFIG,
    'promptfoo_online_config': PROMPTFOO_ONLINE_CONFIG,
    'has_build_input_csv_cmd': bool(BUILD_INPUT_CSV_CMD),
}
show_table([config_row], 'Active configuration')

In [ ]:
# Purpose: display available example tool IDs, process names, and optimization recipes before choosing profiles.
tool_models = [
    {'algorithm': 'dtcrs', 'target_model_id': 'summarizer---dtcrs', 'note': 'example tool/pipe model id'},
    {'algorithm': 'kohaku', 'target_model_id': 'summarizer---kohaku', 'note': 'example tool/pipe model id'},
    {'algorithm': 'kohaku_openrouter', 'target_model_id': 'summarizer---kohaku-OR', 'note': 'example OpenRouter-backed pipe id'},
    {'algorithm': 'lightrag', 'target_model_id': 'summarizer---lightrag', 'note': 'example tool/pipe model id'},
    {'algorithm': 'raptor', 'target_model_id': 'summarizer---raptor', 'note': 'example tool/pipe model id'},
]

PROCESS_TO_METHOD = {'offline': 'step2_promptfoo', 'online': 'step3_promptfoo'}
PROCESS_LABELS = {
    'offline': 'Offline Process (step2_promptfoo)',
    'online': 'Online Process (step3_promptfoo)',
}

method_matrix = [
    {'label': 'Direct DEA', 'method': 'direct_dea', 'generation': 'bridge', 'evaluation': 'native DEA', 'trace': 'bridge trace + total runtime', 'cost': 'medium'},
    {'label': 'Direct DEA judge', 'method': 'direct_dea_judge', 'generation': 'bridge', 'evaluation': 'native DEA + DEA LLM judge', 'trace': 'bridge trace + judge runtime', 'cost': 'high'},
    {'label': 'Direct LLM judge', 'method': 'direct_llm_judge', 'generation': 'bridge', 'evaluation': 'custom JSON LLM judge', 'trace': 'bridge trace + judge runtime', 'cost': 'medium/high'},
    {'label': PROCESS_LABELS['offline'], 'method': PROCESS_TO_METHOD['offline'], 'generation': 'bridge first', 'evaluation': 'PromptFoo scores saved CSV', 'trace': 'bridge trace + PromptFoo artifacts', 'cost': 'high'},
    {'label': PROCESS_LABELS['online'], 'method': PROCESS_TO_METHOD['online'], 'generation': 'PromptFoo live bridge provider', 'evaluation': 'PromptFoo live scoring', 'trace': 'PromptFoo artifacts + command runtime', 'cost': 'high'},
]

optimization_recipes = [
    {'profile': 'model_system_prompt_only', 'target': 'model', 'configure': 'SYSTEM_PROMPT', 'notes': 'Keep MODEL_PARAMS minimal; useful for prompt-focused comparisons.'},
    {'profile': 'model_sampling_params', 'target': 'model', 'configure': 'MODEL_PARAMS', 'notes': 'Uses temperature/top_p/max_tokens convenience params.'},
    {'profile': 'model_extra_params', 'target': 'model', 'configure': 'MODEL_EXTRA_PAYLOAD', 'notes': 'For provider-specific params such as frequency_penalty or think.'},
    {'profile': 'tool_kohaku_example', 'target': 'tool', 'configure': 'TOOL_TARGET_MODEL_ID and TOOL_PARAMS', 'notes': 'Example only; depends on your OpenWebUI tool setup.'},
    {'profile': 'tool_algorithm_grid', 'target': 'tool', 'configure': 'tool_models', 'notes': 'Compares example algorithm-specific pipes.'},
    {'profile': 'promptfoo_model_offline', 'target': 'model', 'configure': 'PROMPTFOO_OFFLINE_CONFIG', 'notes': 'Offline Process: Trace generates answers, PromptFoo scores saved CSV.'},
    {'profile': 'promptfoo_model_online', 'target': 'model', 'configure': 'PROMPTFOO_ONLINE_CONFIG', 'notes': 'Online Process: PromptFoo calls the bridge live.'},
    {'profile': 'promptfoo_tool_offline', 'target': 'tool', 'configure': 'PROMPTFOO_OFFLINE_CONFIG', 'notes': 'Offline Process for a tool/pipe.'},
    {'profile': 'promptfoo_tool_online', 'target': 'tool', 'configure': 'PROMPTFOO_ONLINE_CONFIG', 'notes': 'Online Process for a tool/pipe.'},
    {'profile': 'offline_calibration', 'target': 'model + tool', 'configure': 'none', 'notes': 'Fast fake PromptFoo score, real Trace trainer artifacts, no OpenWebUI call.'},
]

show_table(tool_models, 'Example tool model IDs')
show_table(method_matrix, 'Optimization methods and process names')
show_table(optimization_recipes, 'Recipe profiles')

In [ ]:
# Purpose: define experiment recipes. Edit profiles here only when adding/removing experiments.
def fake_promptfoo_command() -> str:
    """Return a deterministic PromptFoo command for offline plumbing checks."""
    return (
        "python -c \"import json,os,pathlib,sys; "
        "path=pathlib.Path(sys.argv[1]); root=os.environ.get('DEA_REPO_ROOT'); "
        "path=path if path.is_absolute() or not root else pathlib.Path(root) / path; "
        "path.parent.mkdir(parents=True, exist_ok=True); "
        "payload=dict(score=0.73, namedScores=dict(promptfoo_score=0.73)); "
        "path.write_text(json.dumps(payload), encoding='utf-8')\" {result_json}"
    )


def with_extra_payload(params: dict[str, Any], extra_payload: dict[str, Any]) -> dict[str, Any]:
    """Return model params with provider-specific payload keys attached."""
    merged = dict(params)
    merged['model_extra_payload_json'] = dict(extra_payload)
    return merged


def promptfoo_config_for(process: str) -> str:
    """Return the configured PromptFoo config for an Offline or Online Process."""
    if process == 'offline':
        return PROMPTFOO_OFFLINE_CONFIG
    if process == 'online':
        return PROMPTFOO_ONLINE_CONFIG
    raise ValueError(f'Unknown process {process!r}')


def promptfoo_experiment(
    name: str,
    *,
    process: str,
    optimize_target: str,
    target_model_id: str,
    system_prompt: str | None = None,
    model_params: dict[str, Any] | None = None,
    tool_params: dict[str, Any] | None = None,
    promptfoo_config: str | None = None,
    promptfoo_eval_cmd: str | None = None,
    optimizer_mode: str = 'noop',
) -> dict[str, Any]:
    """Build one generic PromptFoo Offline/Online Process experiment."""
    return {
        'name': name,
        'process': process,
        'process_label': PROCESS_LABELS[process],
        'method': PROCESS_TO_METHOD[process],
        'optimize_target': optimize_target,
        'target_model_id': target_model_id,
        'system_prompt': system_prompt or SYSTEM_PROMPT,
        'model_params': model_params or MODEL_PARAMS,
        'tool_params': tool_params or TOOL_PARAMS,
        'promptfoo_config': promptfoo_config or promptfoo_config_for(process),
        'promptfoo_workdir': rel(PROMPTFOO_WORKDIR),
        'promptfoo_eval_cmd': promptfoo_eval_cmd or PROMPTFOO_EVAL_CMD,
        'optimizer_mode': optimizer_mode,
    }


def experiment_rows(profile: str) -> list[dict[str, Any]]:
    """Build experiment definitions for the selected recipe profile."""
    profiles: dict[str, list[dict[str, Any]]] = {
        'model_system_prompt_only': [
            {
                'name': 'model_system_prompt_only',
                'method': 'direct_dea',
                'process_label': 'Direct DEA',
                'optimize_target': 'model',
                'target_model_id': TARGET_MODEL_ID,
                'system_prompt': SYSTEM_PROMPT,
                'model_params': {'generation_temperature': MODEL_PARAMS.get('generation_temperature', 0.2), 'generation_max_tokens': MODEL_PARAMS.get('generation_max_tokens', 4096)},
            }
        ],
        'model_sampling_params': [
            {
                'name': 'model_sampling_params',
                'method': 'direct_dea',
                'process_label': 'Direct DEA',
                'optimize_target': 'model',
                'target_model_id': TARGET_MODEL_ID,
                'system_prompt': SYSTEM_PROMPT,
                'model_params': MODEL_PARAMS,
            }
        ],
        'model_extra_params': [
            {
                'name': 'model_extra_params',
                'method': 'direct_dea',
                'process_label': 'Direct DEA',
                'optimize_target': 'model',
                'target_model_id': TARGET_MODEL_ID,
                'system_prompt': SYSTEM_PROMPT,
                'model_params': with_extra_payload(MODEL_PARAMS, MODEL_EXTRA_PAYLOAD),
            }
        ],
        'tool_kohaku_example': [
            {
                'name': 'tool_kohaku_example',
                'method': 'direct_dea',
                'process_label': 'Direct DEA',
                'optimize_target': 'tool',
                'target_model_id': TOOL_TARGET_MODEL_ID,
                'tool_params': TOOL_PARAMS,
            }
        ],
        'tool_algorithm_grid': [
            {
                'name': f"tool_{item['algorithm']}",
                'method': 'direct_dea',
                'process_label': 'Direct DEA',
                'optimize_target': 'tool',
                'target_model_id': item['target_model_id'],
                'tool_params': {'algorithm': item['algorithm'].replace('_openrouter', ''), 'target_length': TOOL_PARAMS.get('target_length', 'long'), 'structure': TOOL_PARAMS.get('structure', 'thematic')},
            }
            for item in tool_models
        ],
        'promptfoo_model_offline': [
            promptfoo_experiment('promptfoo_model_offline', process='offline', optimize_target='model', target_model_id=TARGET_MODEL_ID, system_prompt=SYSTEM_PROMPT, model_params=MODEL_PARAMS),
        ],
        'promptfoo_model_online': [
            promptfoo_experiment('promptfoo_model_online', process='online', optimize_target='model', target_model_id=TARGET_MODEL_ID, system_prompt=SYSTEM_PROMPT, model_params=MODEL_PARAMS),
        ],
        'promptfoo_tool_offline': [
            promptfoo_experiment('promptfoo_tool_offline', process='offline', optimize_target='tool', target_model_id=TOOL_TARGET_MODEL_ID, tool_params=TOOL_PARAMS),
        ],
        'promptfoo_tool_online': [
            promptfoo_experiment('promptfoo_tool_online', process='online', optimize_target='tool', target_model_id=TOOL_TARGET_MODEL_ID, tool_params=TOOL_PARAMS),
        ],
        'method_matrix': [
            {
                'name': f"method_{item['method']}",
                'method': item['method'],
                'process_label': item['label'],
                'optimize_target': 'tool',
                'target_model_id': TOOL_TARGET_MODEL_ID,
                'tool_params': TOOL_PARAMS,
            }
            if item['method'] not in {PROCESS_TO_METHOD['offline'], PROCESS_TO_METHOD['online']}
            else promptfoo_experiment(
                f"method_{item['method']}",
                process='offline' if item['method'] == PROCESS_TO_METHOD['offline'] else 'online',
                optimize_target='tool',
                target_model_id=TOOL_TARGET_MODEL_ID,
                tool_params=TOOL_PARAMS,
            )
            for item in method_matrix
        ],
        'offline_calibration': [
            promptfoo_experiment(
                'offline_model_calibration',
                process='online',
                optimize_target='model',
                target_model_id=TARGET_MODEL_ID,
                system_prompt='Calibration run only.',
                model_params={'generation_temperature': 0, 'generation_max_tokens': 64, 'model_extra_payload_json': {'frequency_penalty': 0.0, 'think': False}},
                promptfoo_eval_cmd=fake_promptfoo_command(),
                optimizer_mode='noop',
            ),
            promptfoo_experiment(
                'offline_tool_calibration',
                process='online',
                optimize_target='tool',
                target_model_id=TOOL_TARGET_MODEL_ID,
                tool_params={'algorithm': TOOL_PARAMS.get('algorithm', 'kohaku'), 'target_length': 'short', 'structure': TOOL_PARAMS.get('structure', 'thematic')},
                promptfoo_eval_cmd=fake_promptfoo_command(),
                optimizer_mode='noop',
            ),
        ],
    }
    if profile not in profiles:
        raise ValueError(f'Unknown profile {profile!r}. Choose one of: {sorted(profiles)}')
    return profiles[profile]

experiments = experiment_rows(PROFILE)
show_table(experiments, f'Selected experiments: {PROFILE}')

In [ ]:
# Purpose: convert selected experiment dictionaries into `trace_openwebui_dea_skeleton.py` CLI commands.
def base_command(exp: dict[str, Any]) -> list[str]:
    """Convert one experiment definition into a Trace CLI command."""
    output_json = RESULTS_DIR / f"{exp['name']}.json"
    artifact_dir = RESULTS_DIR / exp['name']
    args = [
        'python', rel(SCRIPT),
        '--input-csv', rel(INPUT_CSV),
        '--max-rows', str(exp.get('max_rows', 3)),
        '--batch-size', str(exp.get('batch_size', BATCH_SIZE)),
        '--num-epochs', str(exp.get('num_epochs', NUM_EPOCHS)),
        '--optimization-method', exp['method'],
        '--optimize-target', exp['optimize_target'],
        '--target-model-id', exp['target_model_id'],
        '--bridge-url', exp.get('bridge_url', BRIDGE_URL),
        '--bridge-timeout-seconds', str(exp.get('bridge_timeout_seconds', 600)),
        '--artifact-dir', rel(artifact_dir),
        '--output-json', rel(output_json),
        '--optimizer-mode', exp.get('optimizer_mode', 'noop'),
        '--optimizer-max-tokens', str(exp.get('optimizer_max_tokens', 512)),
    ]
    build_cmd = exp.get('build_input_csv_cmd', BUILD_INPUT_CSV_CMD)
    if build_cmd:
        args += ['--build-input-csv-cmd', build_cmd]
    if exp.get('force_build_input_csv', FORCE_BUILD_INPUT_CSV):
        args += ['--force-build-input-csv']
    if exp['optimize_target'] == 'tool':
        args += ['--initial-tool-params-json', json.dumps(exp.get('tool_params', {}), ensure_ascii=False)]
    else:
        args += [
            '--initial-system-prompt', exp.get('system_prompt', SYSTEM_PROMPT),
            '--initial-model-params-json', json.dumps(exp.get('model_params', MODEL_PARAMS), ensure_ascii=False),
        ]
        if exp.get('model_prompt_mode'):
            args += ['--model-prompt-mode', exp['model_prompt_mode']]
    if exp['method'] in set(PROCESS_TO_METHOD.values()):
        args += [
            '--promptfoo-config', exp.get('promptfoo_config', promptfoo_config_for(exp.get('process', 'offline'))),
            '--promptfoo-workdir', exp.get('promptfoo_workdir', rel(PROMPTFOO_WORKDIR)),
            '--promptfoo-eval-cmd', exp.get('promptfoo_eval_cmd', PROMPTFOO_EVAL_CMD),
        ]
    if exp['method'] == 'direct_llm_judge':
        args += [
            '--judge-base-url', exp.get('judge_base_url', 'http://127.0.0.1:8001/v1'),
            '--judge-api-key', exp.get('judge_api_key', 'dummy'),
            '--judge-model', exp.get('judge_model', 'qwen-laptop-dea:latest'),
            '--judge-timeout-seconds', str(exp.get('judge_timeout_seconds', 180)),
            '--judge-extra-body-json', json.dumps(exp.get('judge_extra_body', {'max_tokens': 192}), ensure_ascii=False),
            '--judge-system-prompt', exp.get('judge_system_prompt', 'Return compact strict JSON only.'),
        ]
    return args

plan_rows = []
for exp in experiments:
    cmd = base_command(exp)
    plan_rows.append({
        'name': exp['name'],
        'process': exp.get('process_label', exp.get('method')),
        'method': exp['method'],
        'optimize_target': exp['optimize_target'],
        'target_model_id': exp['target_model_id'],
        'optimizer_mode': exp.get('optimizer_mode', 'noop'),
        'promptfoo_config': exp.get('promptfoo_config', ''),
        'model_params': exp.get('model_params', ''),
        'tool_params': exp.get('tool_params', ''),
        'command': ' '.join(shlex.quote(part) for part in cmd),
    })

show_table(plan_rows, 'Experiment plan')
if pd is not None:
    pd.DataFrame(plan_rows).to_csv(RESULTS_DIR / 'experiment_plan.csv', index=False)
else:
    (RESULTS_DIR / 'experiment_plan.json').write_text(json.dumps(plan_rows, indent=2, ensure_ascii=False), encoding='utf-8')

In [ ]:
# Purpose: show a standalone model-only Offline Process example command without executing it.
model_offline_example = promptfoo_experiment(
    'example_model_offline_process',
    process='offline',
    optimize_target='model',
    target_model_id=TARGET_MODEL_ID,
    system_prompt=SYSTEM_PROMPT,
    model_params=MODEL_PARAMS,
    optimizer_mode='noop',
)
model_offline_example_row = {
    'name': model_offline_example['name'],
    'process': model_offline_example['process_label'],
    'target_model_id': model_offline_example['target_model_id'],
    'promptfoo_config': model_offline_example['promptfoo_config'],
    'command': ' '.join(shlex.quote(part) for part in base_command(model_offline_example)),
}
show_table([model_offline_example_row], 'Model Optimization Example')

In [ ]:
# Purpose: parse completed run JSON into a compact table. Do not edit this cell to configure optimization.
def summarize_output(path: Path) -> dict[str, Any]:
    """Extract score, timing, and parameter columns from a Trace result JSON."""
    if not path.exists():
        return {}
    payload = json.loads(path.read_text(encoding='utf-8'))
    best = payload.get('best_weighted') or {}
    scores = best.get('mean_scores') or {}
    state = best.get('current_state') or {}
    return {
        'best_scalar_objective': best.get('scalar_objective'),
        'score': scores.get('score'),
        'dea_composite': scores.get('dea_composite'),
        'plan': scores.get('plan'),
        'content': scores.get('content'),
        'resources': scores.get('resources'),
        'length_alignment': scores.get('length_alignment'),
        'execution_duration_s': scores.get('execution_duration_s'),
        'promptfoo_duration_s': scores.get('promptfoo_duration_s'),
        'llm_calls': scores.get('llm_calls'),
        'tool_calls': scores.get('tool_calls'),
        'batch_size': best.get('batch_size'),
        'optimizer_mode': best.get('optimizer_mode'),
        'param_json': state.get('param_json'),
        'model_param_json': state.get('model_param_json'),
        'tool_param_json': state.get('tool_param_json'),
        'system_prompt_chars': len(state.get('system_prompt') or ''),
    }

In [ ]:
# Purpose: optionally execute selected experiments and write score/parameter/timing result tables.
results = []
for exp in experiments:
    cmd = base_command(exp)
    output_json = RESULTS_DIR / f"{exp['name']}.json"
    row = {
        'name': exp['name'],
        'process': exp.get('process_label', exp.get('method')),
        'method': exp['method'],
        'target_model_id': exp['target_model_id'],
        'status': 'planned',
    }
    if RUN_LIVE:
        t0 = time.time()
        env = dict(os.environ, DEA_REPO_ROOT=str(REPO_ROOT), DEA_REPO_PATH=str(REPO_ROOT))
        proc = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env, text=True, capture_output=True, timeout=int(os.environ.get('TRACE_EXPERIMENT_TIMEOUT_SECONDS', '7200')))
        stdout_path = RESULTS_DIR / f"{exp['name']}.stdout.txt"
        stderr_path = RESULTS_DIR / f"{exp['name']}.stderr.txt"
        safe_stdout = proc.stdout.replace(str(REPO_ROOT), '.')
        safe_stderr = proc.stderr.replace(str(REPO_ROOT), '.')
        stdout_path.write_text(safe_stdout, encoding='utf-8')
        stderr_path.write_text(safe_stderr, encoding='utf-8')
        row.update({
            'status': 'ok' if proc.returncode == 0 else 'failed',
            'returncode': proc.returncode,
            'wall_s': time.time() - t0,
            'log_stdout': rel(stdout_path),
            'log_stderr': rel(stderr_path),
            'error_tail': safe_stderr[-1000:] if proc.returncode else '',
        })
        row.update(summarize_output(output_json))
    results.append(row)
    show_table([row], f"Result: {exp['name']}")

show_table(results, 'Summary results')
if pd is not None:
    pd.DataFrame(results).to_csv(RESULTS_DIR / 'experiment_results.csv', index=False)
else:
    (RESULTS_DIR / 'experiment_results.json').write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')